In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np

def count_bids_per_day(input_dir, output_dir=None, plot_file=None, csv_file=None):
    """
    Count bids per day from parquet files and create a time series plot
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files with bid data
    output_dir : str, optional
        Directory to save output files (defaults to input_dir if None)
    plot_file : str, optional
        Filename for the plot (defaults to 'bids_per_day_plot.png')
    csv_file : str, optional
        Filename for the CSV data (defaults to 'bids_per_day_counts.csv')
    """
    print(f"Counting bids per day from files in {input_dir}...")
    
    # Set default output directory and filenames if not provided
    if output_dir is None:
        output_dir = input_dir
    if plot_file is None:
        plot_file = "bids_per_day_plot.png"
    if csv_file is None:
        csv_file = "bids_per_day_counts.csv"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get list of all parquet files
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    file_list.sort()
    
    if not file_list:
        print(f"No parquet files found in {input_dir}")
        return
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Initialize a dictionary to store bid counts per day
    all_counts = {}
    bid_types = {}  # To track different bid types
    
    # Process each file
    for idx, file_path in enumerate(file_list, start=1):
        try:
            # Print progress every 10 files
            if idx % 10 == 0 or idx == 1 or idx == len(file_list):
                print(f"Processing file {idx}/{len(file_list)}: {os.path.basename(file_path)}")
            
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            if 'SETTLEMENTDATE' not in df.columns:
                print(f"  Warning: SETTLEMENTDATE column not found in {os.path.basename(file_path)}")
                continue
            
            # Extract the date part from SETTLEMENTDATE
            df['DATE'] = pd.to_datetime(df['SETTLEMENTDATE']).dt.date
            
            # Count bids per day
            daily_counts = df.groupby('DATE').size()
            
            # Update the overall counts
            for date, count in daily_counts.items():
                date_str = str(date)
                if date_str in all_counts:
                    all_counts[date_str] += count
                else:
                    all_counts[date_str] = count
            
            # Count by bid type if available
            if 'BIDTYPE' in df.columns:
                type_counts = df.groupby(['DATE', 'BIDTYPE']).size().reset_index(name='COUNT')
                for _, row in type_counts.iterrows():
                    date_str = str(row['DATE'])
                    bid_type = row['BIDTYPE']
                    count = row['COUNT']
                    
                    if date_str not in bid_types:
                        bid_types[date_str] = {}
                    
                    if bid_type in bid_types[date_str]:
                        bid_types[date_str][bid_type] += count
                    else:
                        bid_types[date_str][bid_type] = count
        
        except Exception as e:
            print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")
    
    if not all_counts:
        print("No bid data found in the files")
        return
    
    # Convert to DataFrame for easier handling
    dates = sorted(all_counts.keys())
    counts = [all_counts[date] for date in dates]
    
    count_df = pd.DataFrame({
        'date': dates,
        'total_bids': counts
    })
    
    # Convert date strings to datetime for proper sorting
    count_df['date'] = pd.to_datetime(count_df['date'])
    count_df = count_df.sort_values('date')
    
    # Add bid type columns if available
    if bid_types:
        # Find all unique bid types
        all_bid_types = set()
        for date_types in bid_types.values():
            all_bid_types.update(date_types.keys())
        
        # Add columns for each bid type
        for bid_type in all_bid_types:
            count_df[f'bids_{bid_type}'] = count_df['date'].astype(str).map(
                lambda date_str: bid_types.get(date_str, {}).get(bid_type, 0)
            )
    
    # Save to CSV
    csv_path = os.path.join(output_dir, csv_file)
    count_df.to_csv(csv_path, index=False)
    print(f"Saved bid counts to {csv_path}")
    
    # Create plot
    create_time_series_plot(count_df, output_dir, plot_file)
    
    return count_df

def create_time_series_plot(count_df, output_dir, plot_file):
    """
    Create a time series plot of bid counts per day
    """
    plt.figure(figsize=(12, 6))
    
    # Set style
    sns.set_style("whitegrid")
    
    # Plot total bids
    ax = sns.lineplot(
        data=count_df,
        x='date',
        y='total_bids',
        linewidth=2,
        marker='o',
        markersize=4
    )
    
    # Plot bid types if available
    bid_type_columns = [col for col in count_df.columns if col.startswith('bids_')]
    if bid_type_columns:
        # Create a separate plot for bid types
        plt.figure(figsize=(12, 6))
        
        # Melt the dataframe to get bid types in one column
        melt_df = pd.melt(
            count_df, 
            id_vars=['date'], 
            value_vars=bid_type_columns,
            var_name='bid_type', 
            value_name='count'
        )
        
        # Clean up bid type names for legend
        melt_df['bid_type'] = melt_df['bid_type'].str.replace('bids_', '')
        
        # Plot bid types
        sns.lineplot(
            data=melt_df,
            x='date',
            y='count',
            hue='bid_type',
            linewidth=2,
            marker='o',
            markersize=4
        )
        
        plt.title('Number of Bids per Day by Bid Type')
        plt.xlabel('Date')
        plt.ylabel('Number of Bids')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Save bid types plot
        bid_types_plot_path = os.path.join(output_dir, 'bids_per_day_by_type_plot.png')
        plt.savefig(bid_types_plot_path, dpi=300)
        print(f"Saved bid types plot to {bid_types_plot_path}")
        
        # Return to the total bids plot
        plt.figure(1)
    
    # Customize total bids plot
    plt.title('Total Number of Bids per Day')
    plt.xlabel('Date')
    plt.ylabel('Number of Bids')
    plt.xticks(rotation=45)
    
    # Add rolling average
    if len(count_df) > 7:
        rolling_avg = count_df['total_bids'].rolling(window=7).mean()
        plt.plot(count_df['date'], rolling_avg, 'r--', linewidth=2, label='7-day Moving Average')
        plt.legend()
    
    plt.tight_layout()
    
    # Save the plot
    plot_path = os.path.join(output_dir, plot_file)
    plt.savefig(plot_path, dpi=300)
    print(f"Saved plot to {plot_path}")
    
    # Show plot stats
    min_date = count_df['date'].min()
    max_date = count_df['date'].max()
    total_days = (max_date - min_date).days + 1
    days_with_data = len(count_df)
    
    print(f"\nPlot statistics:")
    print(f"Date range: {min_date.date()} to {max_date.date()} ({total_days} days)")
    print(f"Days with bid data: {days_with_data}")
    print(f"Maximum bids in a day: {count_df['total_bids'].max()} on {count_df.loc[count_df['total_bids'].idxmax(), 'date'].date()}")
    print(f"Minimum bids in a day: {count_df['total_bids'].min()} on {count_df.loc[count_df['total_bids'].idxmin(), 'date'].date()}")
    print(f"Average bids per day: {count_df['total_bids'].mean():.1f}")

if __name__ == "__main__":
    # Directory containing the parquet files
    input_directory = "/Volumes/T7/bid-volume-filtered-3"  # Update this to your directory
    output_directory = "/Volumes/T7/bid-analysis"  # Where to save the plot and CSV
    
    # Run the analysis
    bid_counts = count_bids_per_day(
        input_dir=input_directory,
        output_dir=output_directory,
        plot_file="fcas_bids_per_day.png",
        csv_file="fcas_bids_per_day.csv"
    )

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import numpy as np

def count_bids_per_day(input_dir, output_dir=None, plot_file=None, csv_file=None):
    """
    Count bids per day from parquet files and create a time series plot
    
    Parameters:
    -----------
    input_dir : str
        Directory containing the parquet files with bid data
    output_dir : str, optional
        Directory to save output files (defaults to input_dir if None)
    plot_file : str, optional
        Filename for the plot (defaults to 'bids_per_day_plot.png')
    csv_file : str, optional
        Filename for the CSV data (defaults to 'bids_per_day_counts.csv')
    """
    print(f"Counting bids per day from files in {input_dir}...")
    
    # Set default output directory and filenames if not provided
    if output_dir is None:
        output_dir = input_dir
    if plot_file is None:
        plot_file = "bids_per_day_plot.png"
    if csv_file is None:
        csv_file = "bids_per_day_counts.csv"
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get list of all parquet files
    file_list = glob.glob(os.path.join(input_dir, "*.parquet"))
    file_list.sort()
    
    if not file_list:
        print(f"No parquet files found in {input_dir}")
        return
    
    print(f"Found {len(file_list)} parquet files to process")
    
    # Initialize a dictionary to store bid counts per day
    all_counts = {}
    bid_types = {}  # To track different bid types
    
    # Process each file
    for idx, file_path in enumerate(file_list, start=1):
        try:
            # Print progress every 10 files
            if idx % 10 == 0 or idx == 1 or idx == len(file_list):
                print(f"Processing file {idx}/{len(file_list)}: {os.path.basename(file_path)}")
            
            # Read the parquet file
            df = pd.read_parquet(file_path)
            
            if 'SETTLEMENTDATE' not in df.columns:
                print(f"  Warning: SETTLEMENTDATE column not found in {os.path.basename(file_path)}")
                continue
            
            # Extract the date part from SETTLEMENTDATE
            df['DATE'] = pd.to_datetime(df['SETTLEMENTDATE']).dt.date
            
            # Count bids per day
            daily_counts = df.groupby('DATE').size()
            
            # Update the overall counts
            for date, count in daily_counts.items():
                date_str = str(date)
                if date_str in all_counts:
                    all_counts[date_str] += count
                else:
                    all_counts[date_str] = count
            
            # Count by bid type if available
            if 'BIDTYPE' in df.columns:
                type_counts = df.groupby(['DATE', 'BIDTYPE']).size().reset_index(name='COUNT')
                for _, row in type_counts.iterrows():
                    date_str = str(row['DATE'])
                    bid_type = row['BIDTYPE']
                    count = row['COUNT']
                    
                    if date_str not in bid_types:
                        bid_types[date_str] = {}
                    
                    if bid_type in bid_types[date_str]:
                        bid_types[date_str][bid_type] += count
                    else:
                        bid_types[date_str][bid_type] = count
        
        except Exception as e:
            print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")
    
    if not all_counts:
        print("No bid data found in the files")
        return
    
    # Convert to DataFrame for easier handling
    dates = sorted(all_counts.keys())
    counts = [all_counts[date] for date in dates]
    
    count_df = pd.DataFrame({
        'date': dates,
        'total_bids': counts
    })
    
    # Convert date strings to datetime for proper sorting
    count_df['date'] = pd.to_datetime(count_df['date'])
    count_df = count_df.sort_values('date')
    
    # Add bid type columns if available
    if bid_types:
        # Find all unique bid types
        all_bid_types = set()
        for date_types in bid_types.values():
            all_bid_types.update(date_types.keys())
        
        # Add columns for each bid type
        for bid_type in all_bid_types:
            count_df[f'bids_{bid_type}'] = count_df['date'].astype(str).map(
                lambda date_str: bid_types.get(date_str, {}).get(bid_type, 0)
            )
    
    # Save to CSV
    csv_path = os.path.join(output_dir, csv_file)
    count_df.to_csv(csv_path, index=False)
    print(f"Saved bid counts to {csv_path}")
    
    # Create plot
    create_time_series_plot(count_df, output_dir, plot_file)
    
    return count_df

def create_time_series_plot(count_df, output_dir, plot_file):
    """
    Create a time series plot of bid counts per day
    """
    plt.figure(figsize=(12, 6))
    
    # Set style
    sns.set_style("whitegrid")
    
    # Plot total bids
    ax = sns.lineplot(
        data=count_df,
        x='date',
        y='total_bids',
        linewidth=2,
        marker='o',
        markersize=4
    )
    
    # Plot bid types if available
    bid_type_columns = [col for col in count_df.columns if col.startswith('bids_')]
    if bid_type_columns:
        # Create a separate plot for bid types
        plt.figure(figsize=(12, 6))
        
        # Melt the dataframe to get bid types in one column
        melt_df = pd.melt(
            count_df, 
            id_vars=['date'], 
            value_vars=bid_type_columns,
            var_name='bid_type', 
            value_name='count'
        )
        
        # Clean up bid type names for legend
        melt_df['bid_type'] = melt_df['bid_type'].str.replace('bids_', '')
        
        # Plot bid types
        sns.lineplot(
            data=melt_df,
            x='date',
            y='count',
            hue='bid_type',
            linewidth=2,
            marker='o',
            markersize=4
        )
        
        plt.title('Number of Bids per Day by Bid Type')
        plt.xlabel('Date')
        plt.ylabel('Number of Bids')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Save bid types plot
        bid_types_plot_path = os.path.join(output_dir, 'bids_per_day_by_type_plot.png')
        plt.savefig(bid_types_plot_path, dpi=300)
        print(f"Saved bid types plot to {bid_types_plot_path}")
        
        # Return to the total bids plot
        plt.figure(1)
    
    # Customize total bids plot
    plt.title('Total Number of Bids per Day')
    plt.xlabel('Date')
    plt.ylabel('Number of Bids')
    plt.xticks(rotation=45)
    
    # Add rolling average
    if len(count_df) > 7:
        rolling_avg = count_df['total_bids'].rolling(window=7).mean()
        plt.plot(count_df['date'], rolling_avg, 'r--', linewidth=2, label='7-day Moving Average')
        plt.legend()
    
    plt.tight_layout()
    
    # Save the plot
    plot_path = os.path.join(output_dir, plot_file)
    plt.savefig(plot_path, dpi=300)
    print(f"Saved plot to {plot_path}")
    
    # Show plot stats
    min_date = count_df['date'].min()
    max_date = count_df['date'].max()
    total_days = (max_date - min_date).days + 1
    days_with_data = len(count_df)
    
    print(f"\nPlot statistics:")
    print(f"Date range: {min_date.date()} to {max_date.date()} ({total_days} days)")
    print(f"Days with bid data: {days_with_data}")
    print(f"Maximum bids in a day: {count_df['total_bids'].max()} on {count_df.loc[count_df['total_bids'].idxmax(), 'date'].date()}")
    print(f"Minimum bids in a day: {count_df['total_bids'].min()} on {count_df.loc[count_df['total_bids'].idxmin(), 'date'].date()}")
    print(f"Average bids per day: {count_df['total_bids'].mean():.1f}")

if __name__ == "__main__":
    # Directory containing the parquet files
    input_directory = "/Volumes/T7/bid-volume-filtered-3"  # Update this to your directory
    output_directory = "/Volumes/T7/bid-analysis"  # Where to save the plot and CSV
    
    # Run the analysis
    bid_counts = count_bids_per_day(
        input_dir=input_directory,
        output_dir=output_directory,
        plot_file="fcas_bids_per_day.png",
        csv_file="fcas_bids_per_day.csv"
    )